# Camada Silver

Transformação dos dados brutos da camada Bronze em tabelas Silver tratadas e normalizadas

## 1. Setup

Importação de funções do PySpark e criação do schema silver

In [0]:
from pyspark.sql.functions import (
    col, trim, lower, when, regexp_replace, to_date, coalesce, year,
    row_number, expr, split, explode, initcap, last, lit, round as spark_round,
    max as spark_max, try_to_date
)
from pyspark.sql.window import Window
from pyspark.sql.types import DecimalType, IntegerType, DoubleType, DateType

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

DataFrame[]

## 2. Cotação do Dólar

Criação da tabela `silver.tb_cotacao_dolar` com cotações diárias USD to BRL

In [0]:
df_cotacao_base = (
    spark.table("bronze.tb_cotacao_dolar")
    .withColumn("data_cotacao", to_date(col("dataHoraCotacao")))
    .withColumn("taxa_compra", col("cotacaoCompra").cast(DecimalType(10, 4)))
    .filter(col("data_cotacao").isNotNull())
    .dropDuplicates(["data_cotacao"])
)


result = df_cotacao_base.selectExpr("min(data_cotacao)", "max(data_cotacao)").first()
min_data, max_data = result if result else (None, None)

if min_data and max_data:
    df_calendario = spark.sql(f"""
        SELECT explode(sequence(to_date('{min_data}'), to_date('{max_data}'), interval 1 day)) as data_referencia
    """)
    
   
    df_cotacao_silver = (
        df_calendario.alias("c")
        .join(df_cotacao_base.alias("v"), col("c.data_referencia") >= col("v.data_cotacao"), "left")
        .groupBy(col("c.data_referencia"))
        .agg(spark_max("v.data_cotacao").alias("ultima_data_cotacao"))
        .join(df_cotacao_base.alias("v2"), col("ultima_data_cotacao") == col("v2.data_cotacao"), "left")
        .select(
            col("data_referencia").alias("data"),
            col("v2.taxa_compra").cast(DecimalType(10, 4)).alias("taxa_cambio_brl")
        )
    )
    df_cotacao_silver.write.format("delta").mode("overwrite").saveAsTable("silver.tb_cotacao_dolar")


taxa_recente = (
    spark.table("silver.tb_cotacao_dolar")
    .filter(col("taxa_cambio_brl").isNotNull())
    .orderBy(col("data").desc())
    .select("taxa_cambio_brl")
    .first()[0]
)

## 3. Informações dos Filmes

Limpeza e normalização de `bronze.tb_movies_info` -> `silver.tb_info_filmes`

In [0]:
df_info_bronze = spark.table("bronze.tb_movies_info")


df_info_step1 = df_info_bronze.withColumn("status_clean", lower(trim(regexp_replace(col("status"), "[-_]", " "))))

df_info_step2 = df_info_step1.withColumn(
    "status_filme",
    when(col("status_clean") == "released", "Lançado")
    .when(col("status_clean") == "post production", "Pós-Produção")
    .when(col("status_clean") == "in production", "Em Produção")
    .when(col("status_clean") == "planned", "Planejado")
    .when(col("status_clean") == "rumored", "Rumores")
    .when(col("status_clean") == "canceled", "Cancelado")
    .otherwise("Não Informado")
)


df_info_step3 = df_info_step2.withColumn(
    "data_lancamento_parsed",
    coalesce(
        try_to_date(col("release_date"), "yyyy-MM-dd"),
        try_to_date(col("release_date"), "dd/MM/yyyy"),
        try_to_date(col("release_date"), "MM/dd/yyyy"),
        try_to_date(col("release_date"), "yyyy/MM/dd"),
        try_to_date(col("release_date"), "dd-MM-yyyy"),
        try_to_date(col("release_date"), "MM-dd-yyyy")
    )
).withColumn("ano_lancamento", year(col("data_lancamento_parsed")))


w_dedup = Window.partitionBy("id").orderBy(col("ingestion_datetime").desc())

df_info_silver = (
    df_info_step3
    .withColumn("rn", row_number().over(w_dedup))
    .filter(col("rn") == 1)
    .select(
        col("id").alias("id_filme"),
        col("title").alias("titulo"),
        col("original_title").alias("titulo_original"),
        col("data_lancamento_parsed").alias("data_lancamento"),
        col("ano_lancamento").cast(IntegerType()),
        when(col("runtime").rlike("^[0-9]+$"), col("runtime").cast(IntegerType())).otherwise(None).alias("duracao_minutos"),
        col("original_language").alias("idioma_original"),
        col("status_filme"),
        col("overview").alias("sinopse"),
        col("tagline").alias("frase_divulgacao")
    )
)

df_info_silver.write.format("delta").mode("overwrite").saveAsTable("silver.tb_info_filmes")

## 4. Dados Financeiros

Transformação de `bronze.tb_movies_financials` -> `silver.tb_financeiro_filmes`

In [0]:
df_fin_bronze = spark.table("bronze.tb_movies_financials")

def limpar_monetario(c):
    cleaned = regexp_replace(regexp_replace(trim(col(c)), "[$R\\s,]", ""), "[^0-9.]", "")
    return (
        when(lower(trim(col(c))).isin("unknown", "não informado", "null", "none", "") | (cleaned == ""), None)
        .otherwise(cleaned.cast(DecimalType(18, 2)))
    )

df_fin_clean = (
    df_fin_bronze
    .withColumn("budget_val", limpar_monetario("budget"))
    .withColumn("revenue_val", limpar_monetario("revenue"))
    # Valores zerados ou negativos tratados como ausentes (NULL)
    .withColumn("orcamento_usd", when(col("budget_val") <= 0, None).otherwise(col("budget_val")))
    .withColumn("receita_usd", when(col("revenue_val") <= 0, None).otherwise(col("revenue_val")))
)

# Conversão para BRL e Cálculos de Lucro e Margem
taxa_lit = lit(float(taxa_recente))

df_fin_silver = (
    df_fin_clean
    .withColumn("orcamento_brl", (col("orcamento_usd") * taxa_lit).cast(DecimalType(18, 2)))
    .withColumn("receita_brl", (col("receita_usd") * taxa_lit).cast(DecimalType(18, 2)))
    .withColumn("lucro_usd", (col("receita_usd") - col("orcamento_usd")).cast(DecimalType(18, 2)))
    .withColumn("lucro_brl", (col("receita_brl") - col("orcamento_brl")).cast(DecimalType(18, 2)))
    .withColumn(
        "margem_lucro_pct",
        when((col("receita_usd").isNotNull()) & (col("receita_usd") > 0) & (col("lucro_usd").isNotNull()),
             spark_round((col("lucro_usd") / col("receita_usd")) * 100, 2))
        .otherwise(None)
    )
    .select(
        col("id").alias("id_filme"),
        "orcamento_usd",
        "receita_usd",
        "orcamento_brl",
        "receita_brl",
        "lucro_usd",
        "lucro_brl",
        "margem_lucro_pct"
    )
)

df_fin_silver.write.format("delta").mode("overwrite").saveAsTable("silver.tb_financeiro_filmes")

## 5. Métricas de Engajamento

Limpeza e validação de `bronze.tb_movies_metrics` -> `silver.tb_metricas_engajamento`.

In [0]:
df_met_bronze = spark.table("bronze.tb_movies_metrics")

# Limpeza de popularidade e cast seguro de colunas afetadas por Column Shift
df_met_clean = (
    df_met_bronze
    .withColumn("pop_cleaned", regexp_replace(col("popularity"), ",", "."))
    .withColumn("popularity_cast", expr("try_cast(pop_cleaned as double)"))
    .withColumn("vote_avg_cast", expr("try_cast(vote_average as double)"))
    .withColumn("vote_cnt_cast", expr("try_cast(vote_count as int)"))
    .withColumn("imdb_rating_cast", expr("try_cast(averageRating as double)"))
    .withColumn("imdb_votes_cast", expr("try_cast(numVotes as int)"))
)

# Aplicação de limites de negócio (notas 0 a 10; votos e popularidade >= 0)
df_met_silver = (
    df_met_clean
    .withColumn("popularidade", when(col("popularity_cast") < 0, None).otherwise(col("popularity_cast")))
    .withColumn("nota_media_tmdb", when((col("vote_avg_cast") >= 0) & (col("vote_avg_cast") <= 10), col("vote_avg_cast")).otherwise(None))
    .withColumn("qtd_votos_tmdb", when(col("vote_cnt_cast") < 0, None).otherwise(col("vote_cnt_cast")))
    .withColumn("nota_media_imdb", when((col("imdb_rating_cast") >= 0) & (col("imdb_rating_cast") <= 10), col("imdb_rating_cast")).otherwise(None))
    .withColumn("qtd_votos_imdb", when(col("imdb_votes_cast") < 0, None).otherwise(col("imdb_votes_cast")))
    .select(
        col("id").alias("id_filme"),
        "popularidade",
        "nota_media_tmdb",
        "qtd_votos_tmdb",
        "nota_media_imdb",
        "qtd_votos_imdb"
    )
)

df_met_silver.write.format("delta").mode("overwrite").saveAsTable("silver.tb_metricas_engajamento")

## 6. Avaliações de Usuários

Tratamento de `bronze.tb_movies_reviews` -> `silver.tb_avaliacoes_usuarios`

In [0]:
df_rev_bronze = spark.table("bronze.tb_movies_reviews")

df_rev_silver = (
    df_rev_bronze
    .dropDuplicates(["id", "nome", "nota", "comentario"])
    .withColumn("nota_cast", col("nota").cast(DoubleType()))
    .withColumn("nota_usuario", when((col("nota_cast") >= 0) & (col("nota_cast") <= 10), col("nota_cast")).otherwise(None))
    .withColumn(
        "comentario_usuario",
        when((trim(col("comentario")) == "") | (col("comentario").isNull()), "Sem comentário")
        .otherwise(trim(col("comentario")))
    )
    .select(
        col("id").alias("id_filme"),
        col("nome").alias("nome_usuario"),
        "nota_usuario",
        "comentario_usuario"
    )
)

df_rev_silver.write.format("delta").mode("overwrite").saveAsTable("silver.tb_avaliacoes_usuarios")

## 7. Gêneros, Pessoas e Empresas

Exploração de `bronze.tb_credits_and_tags` em duas tabelas Silver:

- `silver.tb_generos` — gêneros explodidos e normalizados (split por vírgula/ponto-e-vírgula)
- `silver.tb_pessoas_empresas` — atores, diretores, roteiristas e produtoras unificados com tipo de entidade

In [0]:
df_credits = spark.table("bronze.tb_credits_and_tags")


df_generos_silver = (
    df_credits
    .withColumn("genres_norm", regexp_replace(col("genres"), ";", ","))
    .withColumn("genero_item", explode(split(col("genres_norm"), ",")))
    .withColumn("nome_genero", trim(col("genero_item")))
    # Filtro contra ruídos/números decorrentes de column shift
    .filter((col("nome_genero").isNotNull()) & (col("nome_genero") != "") & (~col("nome_genero").rlike("^[0-9]+$")))
    .select(
        col("id").alias("id_filme"),
        initcap(col("nome_genero")).alias("nome_genero")
    )
    .dropDuplicates()
)

df_generos_silver.write.format("delta").mode("overwrite").saveAsTable("silver.tb_generos")


def explodir_entidades(col_origem, tipo_entidade):
    return (
        df_credits
        .withColumn("entidade_norm", regexp_replace(col(col_origem), ";", ","))
        .withColumn("entidade_item", explode(split(col("entidade_norm"), ",")))
        .withColumn("nome_entidade", trim(col("entidade_item")))
        .filter((col("nome_entidade").isNotNull()) & (col("nome_entidade") != "") & (~col("nome_entidade").rlike("^[0-9.]+$")))
        .select(
            col("id").alias("id_filme"),
            initcap(col("nome_entidade")).alias("nome_entidade"),
            lit(tipo_entidade).alias("tipo_entidade")
        )
    )

df_atores = explodir_entidades("cast", "Ator")
df_diretores = explodir_entidades("directors", "Diretor")
df_roteiristas = explodir_entidades("writers", "Roteirista")
df_produtoras = explodir_entidades("production_companies", "Produtora")

df_pessoas_empresas_silver = (
    df_atores
    .unionByName(df_diretores)
    .unionByName(df_roteiristas)
    .unionByName(df_produtoras)
    .dropDuplicates()
)

df_pessoas_empresas_silver.write.format("delta").mode("overwrite").saveAsTable("silver.tb_pessoas_empresas")